In [1]:
import polars as pl

train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')

train.head()

id,prompt,answer
str,str,str
"""00066667""","""In Alice's Wonderland, a secre…","""10010111"""
"""000b53cf""","""In Alice's Wonderland, a secre…","""01000011"""
"""00189f6a""","""In Alice's Wonderland, secret …","""cat imagines book"""
"""001b24c4""","""In Alice's Wonderland, numbers…","""XXXVIII"""
"""001c63cb""","""In Alice's Wonderland, secret …","""wizard creates secret"""


In [2]:
import site

cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)

import kagglehub
import mamba_ssm
import torch
import os
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, TaskType, prepare_model_for_kbit_training 
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, get_cosine_schedule_with_warmup
import shutil
# Configuration

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32  # Can be set to a maximum of 32
OFFLOAD_DIR = os.path.join(OUTPUT_DIR, "offload")
os.makedirs(OFFLOAD_DIR, exist_ok=True)
shutil.rmtree(OFFLOAD_DIR)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
print("Model loaded successfully.")

# Initialize LoRA Adapter
print(f"Initializing LoRA adapter with rank={LORA_RANK}...")
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# YOUR CODE HERE
# --------------
# model.train() 
# --------------
TRAIN_CSV       = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"
SFT_EPOCHS      = 3
SFT_LR          = 2e-4
SFT_GRAD_ACCUM  = 8
SFT_MAX_SEQ_LEN = 1024
WARMUP_RATIO    = 0.1

SYSTEM_PROMPT = (
    "You are an expert reasoning assistant. "
    "Carefully analyze the pattern or rule shown in the examples, "
    "then determine the answer for the given input. "
    "Think step by step, then place your final answer inside \\boxed{}."
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def build_prompt(user_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_text},
    ]
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    return (
        f"<extra_id_0>System\n{SYSTEM_PROMPT}\n"
        f"<extra_id_1>User\n{user_text}\n"
        f"<extra_id_1>Assistant\n"
    )

def build_sft_example(user_text, answer):
    return (
        build_prompt(user_text)
        + "Let me work through this carefully.\n\n"
          "After analyzing the pattern:\n"
        + f"The answer is \\boxed{{{answer}}}"
        + tokenizer.eos_token
    )

class SFTDataset(Dataset):
    def __init__(self, csv_path, max_len):
        df = pl.read_csv(csv_path)
        self.items = []
        for row in df.iter_rows(named=True):
            full_text = build_sft_example(row["prompt"], row["answer"])
            input_ids = tokenizer(
                full_text,
                truncation=True,
                max_length=max_len,
                return_tensors="pt"
            ).input_ids[0]

            prompt_ids = tokenizer(
                build_prompt(row["prompt"]),
                truncation=True,
                max_length=max_len
            ).input_ids
            prompt_len = len(prompt_ids)

            labels = input_ids.clone()
            labels[:prompt_len] = -100
            self.items.append({"input_ids": input_ids, "labels": labels})

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]

def collate(batch):
    max_len = max(x["input_ids"].size(0) for x in batch)
    B = len(batch)

    ids  = torch.full((B, max_len), tokenizer.pad_token_id, dtype=torch.long)
    lbs  = torch.full((B, max_len), -100, dtype=torch.long)
    mask = torch.zeros(B, max_len, dtype=torch.long)

    for i, x in enumerate(batch):
        n = x["input_ids"].size(0)
        ids[i, :n]  = x["input_ids"]
        lbs[i, :n]  = x["labels"]
        mask[i, :n] = 1

    return {"input_ids": ids, "attention_mask": mask, "labels": lbs}

def get_input_device(model):
    # for device_map="auto", send inputs to the first non-cpu/non-disk device
    if hasattr(model, "hf_device_map"):
        for dev in model.hf_device_map.values():
            if dev not in ("cpu", "disk"):
                return torch.device(dev)
    return next(model.parameters()).device

dataset = SFTDataset(TRAIN_CSV, SFT_MAX_SEQ_LEN)
loader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate)

model.train()
input_device = get_input_device(model)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=SFT_LR,
    weight_decay=0.01,
)

total_steps = math.ceil(len(loader) / SFT_GRAD_ACCUM) * SFT_EPOCHS
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * WARMUP_RATIO),
    num_training_steps=total_steps,
)

optimizer.zero_grad()

for epoch in range(1, SFT_EPOCHS + 1):
    epoch_loss = 0.0

    for step, batch in enumerate(loader):
        batch = {k: v.to(input_device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss / SFT_GRAD_ACCUM
        loss.backward()

        epoch_loss += loss.item() * SFT_GRAD_ACCUM

        if (step + 1) % SFT_GRAD_ACCUM == 0 or (step + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    print(f"Epoch {epoch}/{SFT_EPOCHS} — avg loss: {epoch_loss / len(loader):.4f}")

model.eval()


# Save Adapter
print(f"Saving adapter to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/offload'

In [ ]:
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)

In [ ]:
print('Done.')